# Preliminary Data Cleaning

This notebook performs the initial cleaning steps on the TMDB and Trakt datasets, including duplicate removal, schema harmonization, and rating normalization.

In [1]:
import pandas as pd
import numpy as np
import re
import json
import ast
from pathlib import Path
from dataclasses import dataclass

BASE_DIR = Path.cwd().parent.parent
DATA_DIR = BASE_DIR / "data"
CLEANED_DIR = DATA_DIR / "cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Cleaning Logic Definitions

In [2]:
COMMON_SCHEMA_COLUMNS = [
    "user_id", "movie_id", "movie_title", "rating", "genres", "cast",
    "release_year", "language", "release_year_clean", "genres_list",
    "primary_genre", "genre_count",
]

def _canonicalize_genre_name(name: str) -> str:
    normalized = name.strip().lower().replace("_", " ").replace("-", " ")
    normalized = re.sub(r"\s+", " ", normalized)
    alias_map = {"sci fi": "science fiction", "sci fi fantasy": "science fiction"}
    return alias_map.get(normalized, normalized)

def parse_genres(value) -> list[str]:
    if isinstance(value, (list, tuple, set)):
        return [_canonicalize_genre_name(str(item["name"])) if isinstance(item, dict) else _canonicalize_genre_name(str(item)) for item in value if item]
    if pd.isna(value): return []
    raw = str(value).strip()
    if "|" in raw: return [_canonicalize_genre_name(p.strip()) for p in raw.split("|") if p.strip()]
    return [_canonicalize_genre_name(p.strip()) for p in re.split(r"[,;/]", raw) if p.strip()]

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ["rating", "vote_average"]: 
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors="coerce")
    if "release_year" in df.columns:
        year_num = pd.to_numeric(df["release_year"], errors="coerce")
        df["release_year_clean" ] = year_num.where(year_num.between(1870, 2100), np.nan).astype("Int64")
    if "genres" in df.columns:
        df["genres_list"] = df["genres"].apply(parse_genres)
        df["primary_genre"] = df["genres_list"].apply(lambda x: x[0] if x else None)
        df["genre_count"] = df["genres_list"].apply(len)
    return df

def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df = standardize_columns(df.copy())
    # Harmonize schema
    if "cast_names" in df.columns: df = df.rename(columns={"cast_names": "cast"})
    for col in COMMON_SCHEMA_COLUMNS:
        if col not in df.columns: df[col] = pd.NA
    df = df[COMMON_SCHEMA_COLUMNS].copy()
    
    # Remove duplicates
    df = df.drop_duplicates(subset=["user_id", "movie_id"], keep="last")
    
    # Drop rows missing core info
    df = df.dropna(subset=["user_id", "movie_id", "rating"])
    
    # Filter English
    if "language" in df.columns: df = df[df["language"].str.lower() == "en"]
    
    return df

def adjust_half_step_ratings(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rating = pd.to_numeric(df["rating"], errors="coerce")
    frac = rating - np.floor(rating)
    mask = np.isclose(frac, 0.5)
    df.loc[mask, "rating"] = rating.loc[mask] + 0.5
    return df

## 2. Process TMDB Dataset

In [3]:
tmdb_raw = pd.read_csv(DATA_DIR / "movie_final_dataset.csv")
tmdb_cleaned = clean_dataset(tmdb_raw)
tmdb_cleaned = adjust_half_step_ratings(tmdb_cleaned)

tmdb_cleaned.to_csv(CLEANED_DIR / "movie_final_dataset_cleaned.csv", index=False)

print("### TMDB Cleaned Head ###")
display(tmdb_cleaned.head())
print("\n### TMDB Cleaned Info ###")
tmdb_cleaned.info()
print("\n### TMDB Cleaned Describe ###")
display(tmdb_cleaned.describe())

### TMDB Cleaned Head ###


,user_id,movie_id,movie_title,rating,genres,cast,release_year,language,release_year_clean,genres_list,primary_genre,genre_count
360,MontyTG,1228246,Five Nights at Freddy's 2,8.0,Horror|Thriller,"Josh Hutcherson, Piper Rubio, Elizabeth Lail",2025,en,2025,"[horror, thriller]",horror,2
2774,Bradley's,4247,Scary Movie,10.0,Comedy,"Anna Faris, Regina Hall, Marlon Wayans",2000,en,2000,[comedy],comedy,1
2775,katch22,4247,Scary Movie,7.0,Comedy,"Anna Faris, Regina Hall, Marlon Wayans",2000,en,2000,[comedy],comedy,1
3160,Wuchak,45881,The Bridge of San Luis Rey,7.0,Romance|Drama,"Gabriel Byrne, F. Murray Abraham, Kathy Bates",2004,en,2004,"[romance, drama]",romance,2
3185,r96sk,746041,A Nashville Christmas Carol,6.0,TV Movie|Romance,"Jessy Schram, Wes Brown, Kix Brooks",2020,en,2020,"[tv movie, romance]",tv movie,2



### TMDB Cleaned Info ###
<class 'pandas.DataFrame'>
Index: 5335 entries, 360 to 9999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   user_id             5335 non-null   str    
 1   movie_id            5335 non-null   int64  
 2   movie_title         5335 non-null   str    
 3   rating              5335 non-null   float64
 4   genres              5335 non-null   str    
 5   cast                5335 non-null   str    
 6   release_year        5335 non-null   int64  
 7   language            5335 non-null   str    
 8   release_year_clean  5335 non-null   Int64  
 9   genres_list         5335 non-null   object 
 10  primary_genre       5335 non-null   str    
 11  genre_count         5335 non-null   int64  
dtypes: Int64(1), float64(1), int64(3), object(1), str(6)
memory usage: 1.1+ MB

### TMDB Cleaned Describe ###


,movie_id,rating,release_year,release_year_clean,genre_count
count,5.335000e+03,5335.000000,5335.000000,5335.0,5335.000000
mean,3.693476e+05,6.890909,2011.326336,2011.326336,2.827366
std,3.854775e+05,2.119530,15.493391,15.493391,0.836907
min,1.100000e+01,1.000000,1927.000000,1927.0,1.000000
25%,1.002000e+04,6.000000,2005.000000,2005.0,2.000000
50%,2.936600e+05,7.000000,2017.000000,2017.0,3.000000
75%,5.936430e+05,8.000000,2023.000000,2023.0,3.000000
max,1.584215e+06,10.000000,2026.000000,2026.0,6.000000


## 3. Process Trakt Dataset

In [4]:
trakt_raw = pd.read_csv(DATA_DIR / "trakt_ultimate_checkpoint.csv")
trakt_cleaned = clean_dataset(trakt_raw)
trakt_cleaned = adjust_half_step_ratings(trakt_cleaned)

trakt_cleaned.to_csv(CLEANED_DIR / "trakt_ultimate_checkpoint_cleaned.csv", index=False)

print("### Trakt Cleaned Head ###")
display(trakt_cleaned.head())
print("\n### Trakt Cleaned Info ###")
trakt_cleaned.info()
print("\n### Trakt Cleaned Describe ###")
display(trakt_cleaned.describe())

### Trakt Cleaned Head ###


,user_id,movie_id,movie_title,rating,genres,cast,release_year,language,release_year_clean,genres_list,primary_genre,genre_count
0,trevisanb,853702,Superman,8,"superhero, science-fiction, action, adventure","David Corenswet, Rachel Brosnahan, Nicholas Ho...",2025,en,2025,"[superhero, science fiction, action, adventure]",superhero,4
1,jackiejill456,853702,Superman,8,"superhero, science-fiction, action, adventure","David Corenswet, Rachel Brosnahan, Nicholas Ho...",2025,en,2025,"[superhero, science fiction, action, adventure]",superhero,4
2,Klariis,853702,Superman,7,"superhero, science-fiction, action, adventure","David Corenswet, Rachel Brosnahan, Nicholas Ho...",2025,en,2025,"[superhero, science fiction, action, adventure]",superhero,4
3,WebSorve,853702,Superman,6,"superhero, science-fiction, action, adventure","David Corenswet, Rachel Brosnahan, Nicholas Ho...",2025,en,2025,"[superhero, science fiction, action, adventure]",superhero,4
4,Scoop20906,853702,Superman,9,"superhero, science-fiction, action, adventure","David Corenswet, Rachel Brosnahan, Nicholas Ho...",2025,en,2025,"[superhero, science fiction, action, adventure]",superhero,4



### Trakt Cleaned Info ###
<class 'pandas.DataFrame'>
Index: 5223 entries, 0 to 11499
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   user_id             5223 non-null   str   
 1   movie_id            5223 non-null   int64 
 2   movie_title         5223 non-null   str   
 3   rating              5223 non-null   int64 
 4   genres              5187 non-null   str   
 5   cast                5111 non-null   str   
 6   release_year        5223 non-null   int64 
 7   language            5223 non-null   str   
 8   release_year_clean  5223 non-null   Int64 
 9   genres_list         5223 non-null   object
 10  primary_genre       5187 non-null   str   
 11  genre_count         5223 non-null   int64 
dtypes: Int64(1), int64(4), object(1), str(6)
memory usage: 1.4+ MB

### Trakt Cleaned Describe ###


,movie_id,rating,release_year,release_year_clean,genre_count
count,5.223000e+03,5223.000000,5223.000000,5223.0,5223.000000
mean,1.026921e+06,5.598698,2025.103006,2025.103006,2.285277
std,2.386123e+05,2.478049,0.924224,0.924224,1.009998
min,7.952000e+03,1.000000,1999.000000,1999.0,0.000000
25%,9.010470e+05,4.000000,2025.000000,2025.0,2.000000
50%,1.048115e+06,6.000000,2025.000000,2025.0,2.000000
75%,1.200315e+06,7.000000,2025.000000,2025.0,3.000000
max,1.361720e+06,10.000000,2026.000000,2026.0,7.000000
